# UC05 - Operasional Crushing Plant
## Bronze to Silver/Gold Pipeline

Notebook ini digunakan untuk membuat pipeline sederhana dari tabel source/bronze:

```sql
`databricks-phase-a`.`default`.`synova_transaction`
```

Menjadi tabel Silver dan Gold untuk kebutuhan dashboard:

- Monitoring kendaraan masuk/keluar CP
- Total transaksi kendaraan
- Total tonnage
- Durasi antrean dan durasi di area CP
- Performa lane
- Exception/anomaly monitoring

## 0. Parameter Pipeline

Ubah parameter melalui widget Databricks, atau edit nilai default di bawah.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

DEFAULT_CATALOG = "uc"
DEFAULT_SCHEMA = "synova"
DEFAULT_BRONZE_TABLE = "synova_transaction"

DEFAULT_SILVER_TABLE = "silver_uc05_cp_vehicle_movement"
DEFAULT_GOLD_DAILY_TABLE = "gold_uc05_cp_vehicle_daily_summary"
DEFAULT_GOLD_HOURLY_TABLE = "gold_uc05_cp_vehicle_hourly_summary"
DEFAULT_GOLD_LANE_TABLE = "gold_uc05_cp_vehicle_lane_summary"
DEFAULT_GOLD_EXCEPTION_TABLE = "gold_uc05_cp_vehicle_exception_monitoring"


def _get_widget_value(name: str, default: str) -> str:
    """Create/read Databricks widget with safe fallback for non-Databricks execution."""
    try:
        dbutils.widgets.text(name, default)  # type: ignore[name-defined]
        return dbutils.widgets.get(name)    # type: ignore[name-defined]
    except Exception:
        return default


catalog = _get_widget_value("catalog", DEFAULT_CATALOG)
schema = _get_widget_value("schema", DEFAULT_SCHEMA)
bronze_table = _get_widget_value("bronze_table", DEFAULT_BRONZE_TABLE)

silver_table = _get_widget_value("silver_table", DEFAULT_SILVER_TABLE)
gold_daily_table = _get_widget_value("gold_daily_table", DEFAULT_GOLD_DAILY_TABLE)
gold_hourly_table = _get_widget_value("gold_hourly_table", DEFAULT_GOLD_HOURLY_TABLE)
gold_lane_table = _get_widget_value("gold_lane_table", DEFAULT_GOLD_LANE_TABLE)
gold_exception_table = _get_widget_value("gold_exception_table", DEFAULT_GOLD_EXCEPTION_TABLE)


def quote_identifier(value: str) -> str:
    """Quote catalog/schema/table names safely, especially catalog with dash."""
    return f"`{value.replace('`', '``')}`"


def fqn(table_name: str) -> str:
    """Return fully qualified table name: `catalog`.`schema`.`table`."""
    return ".".join([
        quote_identifier(catalog),
        quote_identifier(schema),
        quote_identifier(table_name),
    ])


BRONZE_FQN = fqn(bronze_table)
SILVER_FQN = fqn(silver_table)
GOLD_DAILY_FQN = fqn(gold_daily_table)
GOLD_HOURLY_FQN = fqn(gold_hourly_table)
GOLD_LANE_FQN = fqn(gold_lane_table)
GOLD_EXCEPTION_FQN = fqn(gold_exception_table)

print("Pipeline configuration:")
print(f"Bronze/source : {BRONZE_FQN}")
print(f"Silver        : {SILVER_FQN}")
print(f"Gold daily    : {GOLD_DAILY_FQN}")
print(f"Gold hourly   : {GOLD_HOURLY_FQN}")
print(f"Gold lane     : {GOLD_LANE_FQN}")
print(f"Gold exception: {GOLD_EXCEPTION_FQN}")

## 1. Helper Function

Helper ini dibuat agar notebook tetap aman jika ada kolom yang belum tersedia di source table.

In [0]:

def clean_backslash_n(df: DataFrame) -> DataFrame:
    """Convert string literal '\\N' into NULL for all columns."""
    return df.select([
        F.when(F.trim(F.col(c).cast("string")) == "\\N", F.lit(None))
        .otherwise(F.col(c))
        .alias(c)
        for c in df.columns
    ])


def has_col(df: DataFrame, name: str) -> bool:
    return name in df.columns


def safe_col(df: DataFrame, name: str, fallback_type: str = "string"):
    """Return column if exists, otherwise NULL with fallback type."""
    if has_col(df, name):
        return F.col(name)
    return F.lit(None).cast(fallback_type)


def parse_timestamp(df: DataFrame, name: str):
    """Parse several timestamp formats commonly found in CP transaction data."""
    raw = safe_col(df, name).cast("string")
    return F.coalesce(
        F.to_timestamp(raw),
        F.to_timestamp(raw, "yyyy-MM-dd HH:mm:ss.SSSSSSX"),
        F.to_timestamp(raw, "yyyy-MM-dd HH:mm:ss.SSSX"),
        F.to_timestamp(raw, "yyyy-MM-dd HH:mm:ssX"),
        F.to_timestamp(raw, "M/d/yyyy H:mm"),
        F.to_timestamp(raw, "M/d/yyyy H:mm:ss"),
    )


def parse_boolean(df: DataFrame, name: str):
    """Parse boolean values from boolean/string source."""
    raw = F.lower(F.trim(safe_col(df, name).cast("string")))
    return (
        F.when(raw.isin("true", "t", "1", "yes", "y"), F.lit(True))
        .when(raw.isin("false", "f", "0", "no", "n"), F.lit(False))
        .otherwise(F.lit(None).cast("boolean"))
    )


def parse_double(df: DataFrame, name: str):
    """Parse numeric fields. Removes thousand separators if any."""
    raw = F.regexp_replace(safe_col(df, name).cast("string"), ",", "")
    return raw.cast("double")


def parse_long(df: DataFrame, name: str):
    raw = F.regexp_replace(safe_col(df, name).cast("string"), ",", "")
    return raw.cast("long")


def minute_diff(end_col, start_col):
    """Return duration in minutes. Negative duration becomes NULL."""
    return F.when(
        end_col.isNotNull() & start_col.isNotNull() & (end_col >= start_col),
        (F.unix_timestamp(end_col) - F.unix_timestamp(start_col)) / F.lit(60.0)
    ).otherwise(F.lit(None).cast("double"))

## 2. Read Bronze / Source Table

In [0]:
bronze_df = spark.table(BRONZE_FQN)

print(f"Total source rows: {bronze_df.count():,}")
print("Source columns:")
print(bronze_df.columns)

display(bronze_df.limit(10))

## 3. Bronze to Silver

Transformasi utama:

1. Convert `\\N` menjadi `NULL`
2. Standardisasi timestamp
3. Standardisasi numeric dan boolean
4. Buat status kendaraan masuk/keluar
5. Hitung durasi antrean dan durasi di area CP
6. Buat data quality status

In [0]:
from pyspark.sql import functions as F

# Helper khusus timestamp multi-format
def parse_timestamp_multi(df, column_name):
    """
    Parse timestamp dengan beberapa kemungkinan format.
    Jika format tidak valid, return NULL, bukan error.
    """
    if column_name not in df.columns:
        return F.lit(None).cast("timestamp")

    raw_col = F.trim(F.col(column_name).cast("string"))

    normalized_col = (
        F.when(raw_col.isNull(), None)
        .when(raw_col == "", None)
        .when(raw_col == "\\N", None)
        .otherwise(raw_col)
    )

    return F.coalesce(
        # Default parser, untuk format timestamp normal
        F.try_to_timestamp(normalized_col),

        # Format: 6/1/26 6:02
        F.try_to_timestamp(normalized_col, F.lit("M/d/yy H:mm")),
        F.try_to_timestamp(normalized_col, F.lit("M/d/yy HH:mm")),

        # Format: 6/1/26 6:02:30
        F.try_to_timestamp(normalized_col, F.lit("M/d/yy H:mm:ss")),
        F.try_to_timestamp(normalized_col, F.lit("M/d/yy HH:mm:ss")),

        # Format: 6/1/2026 6:02
        F.try_to_timestamp(normalized_col, F.lit("M/d/yyyy H:mm")),
        F.try_to_timestamp(normalized_col, F.lit("M/d/yyyy HH:mm")),

        # Format: 6/1/2026 6:02:30
        F.try_to_timestamp(normalized_col, F.lit("M/d/yyyy H:mm:ss")),
        F.try_to_timestamp(normalized_col, F.lit("M/d/yyyy HH:mm:ss")),

        # Format: 2026-06-24 06:25:05
        F.try_to_timestamp(normalized_col, F.lit("yyyy-MM-dd HH:mm:ss")),

        # Format: 2026-06-24 06:25:05.376713
        F.try_to_timestamp(normalized_col, F.lit("yyyy-MM-dd HH:mm:ss.SSSSSS")),

        # Format: 2026-06-24 06:25:05.376713+08
        F.try_to_timestamp(normalized_col, F.lit("yyyy-MM-dd HH:mm:ss.SSSSSSX")),

        # Format: 2026-06-24 06:25:05.376713+08:00
        F.try_to_timestamp(normalized_col, F.lit("yyyy-MM-dd HH:mm:ss.SSSSSSXXX"))
    )


clean_df = clean_backslash_n(bronze_df)

# Timestamp columns
created_at_ts = parse_timestamp_multi(clean_df, "created_at")
auditupdate_ts = parse_timestamp_multi(clean_df, "auditupdate")
entrance_time_ts = parse_timestamp_multi(clean_df, "entrance_time")
exit_time_ts = parse_timestamp_multi(clean_df, "exit_time")
exit_cp_time_ts = parse_timestamp_multi(clean_df, "exit_cp_time")
assignment_time_ts = parse_timestamp_multi(clean_df, "assignment_time")
rfid_queue_time_ts = parse_timestamp_multi(clean_df, "rfid_queue_time")
exit_antrian_cp_time_ts = parse_timestamp_multi(clean_df, "exit_antrian_cp_time")
entrance_acp_time_ts = parse_timestamp_multi(clean_df, "entrance_acp_time")
rfid_cp_time_ts = parse_timestamp_multi(clean_df, "rfid_cp_time")
geofence_cp_time_ts = parse_timestamp_multi(clean_df, "geofence_cp_time")
entrance_cop_time_ts = parse_timestamp_multi(clean_df, "entrance_cop_time")
rfid_cp_out_time_ts = parse_timestamp_multi(clean_df, "rfid_cp_out_time")
sicantik_time_ts = parse_timestamp_multi(clean_df, "sicantik_time")

# Vehicle in/out time uses fallback timestamp so movement monitoring still works when one signal is missing.
vehicle_in_time_ts = F.coalesce(
    entrance_time_ts,
    entrance_acp_time_ts,
    rfid_cp_time_ts,
    geofence_cp_time_ts,
    assignment_time_ts
)

vehicle_out_time_ts = F.coalesce(
    exit_cp_time_ts,
    exit_time_ts,
    rfid_cp_out_time_ts
)

silver_df = (
    clean_df
    # Standard IDs and numeric fields
    .withColumn("assignment_id_value", parse_long(clean_df, "assignment_id"))
    .withColumn("truck_id_value", parse_long(clean_df, "truck_id"))
    .withColumn("lane_id_value", parse_long(clean_df, "lane_id"))
    .withColumn("lane_id_sb_value", parse_long(clean_df, "lane_id_sb"))
    .withColumn("cp_queue_id_value", parse_long(clean_df, "cp_queue_id"))
    .withColumn("tonnage_value", parse_double(clean_df, "tonnage"))
    .withColumn("round_robin_cp_value", parse_long(clean_df, "round_robin_cp"))

    # Standard boolean fields
    .withColumn("back_to_queue_bool", parse_boolean(clean_df, "back_to_queue"))
    .withColumn("is_anomaly_bool", parse_boolean(clean_df, "is_anomaly"))
    .withColumn("is_anomaly_queue_lane_bool", parse_boolean(clean_df, "is_anomaly_queue_lane"))
    .withColumn("is_northstockpile_2_bool", parse_boolean(clean_df, "is_northstockpile_2"))

    # Standard timestamp fields
    .withColumn("created_at_ts", created_at_ts)
    .withColumn("auditupdate_ts", auditupdate_ts)
    .withColumn("entrance_time_ts", entrance_time_ts)
    .withColumn("exit_time_ts", exit_time_ts)
    .withColumn("exit_cp_time_ts", exit_cp_time_ts)
    .withColumn("assignment_time_ts", assignment_time_ts)
    .withColumn("rfid_queue_time_ts", rfid_queue_time_ts)
    .withColumn("exit_antrian_cp_time_ts", exit_antrian_cp_time_ts)
    .withColumn("entrance_acp_time_ts", entrance_acp_time_ts)
    .withColumn("rfid_cp_time_ts", rfid_cp_time_ts)
    .withColumn("geofence_cp_time_ts", geofence_cp_time_ts)
    .withColumn("entrance_cop_time_ts", entrance_cop_time_ts)
    .withColumn("rfid_cp_out_time_ts", rfid_cp_out_time_ts)
    .withColumn("sicantik_time_ts", sicantik_time_ts)

    # Canonical movement timestamp
    .withColumn("vehicle_in_time_ts", vehicle_in_time_ts)
    .withColumn("vehicle_out_time_ts", vehicle_out_time_ts)

    # Date dimension
    .withColumn(
        "transaction_ts",
        F.coalesce(
            F.col("created_at_ts"),
            F.col("vehicle_in_time_ts"),
            F.col("assignment_time_ts"),
            F.col("auditupdate_ts")
        )
    )
    .withColumn("transaction_date", F.to_date("transaction_ts"))
    .withColumn("transaction_hour", F.hour("transaction_ts"))
)

# Movement status
silver_df = (
    silver_df
    .withColumn(
        "vehicle_movement_status",
        F.when(
            F.col("vehicle_in_time_ts").isNotNull() & F.col("vehicle_out_time_ts").isNotNull(),
            F.lit("IN_OUT_COMPLETED")
        )
        .when(
            F.col("vehicle_in_time_ts").isNotNull() & F.col("vehicle_out_time_ts").isNull(),
            F.lit("IN_NOT_OUT")
        )
        .when(
            F.col("vehicle_in_time_ts").isNull() & F.col("vehicle_out_time_ts").isNotNull(),
            F.lit("OUT_WITHOUT_IN")
        )
        .otherwise(F.lit("UNKNOWN"))
    )
)

# Duration metrics
silver_df = (
    silver_df
    .withColumn(
        "queue_duration_minutes",
        minute_diff(F.col("exit_antrian_cp_time_ts"), F.col("rfid_queue_time_ts"))
    )
    .withColumn(
        "cp_area_duration_minutes",
        minute_diff(F.col("vehicle_out_time_ts"), F.col("vehicle_in_time_ts"))
    )
    .withColumn(
        "cp_process_duration_minutes",
        minute_diff(
            F.col("rfid_cp_out_time_ts"),
            F.coalesce(F.col("rfid_cp_time_ts"), F.col("geofence_cp_time_ts"))
        )
    )
    .withColumn(
        "audit_latency_minutes",
        minute_diff(F.col("auditupdate_ts"), F.col("created_at_ts"))
    )
)

# Data quality status
silver_df = (
    silver_df
    .withColumn(
        "data_quality_status",
        F.when(F.col("assignment_id_value").isNull(), F.lit("missing_assignment_id"))
        .when(safe_col(silver_df, "queue_name").isNull(), F.lit("missing_cp"))
        .when(F.col("truck_id_value").isNull(), F.lit("missing_truck_id"))
        .when(F.col("vehicle_in_time_ts").isNull(), F.lit("missing_vehicle_in_time"))
        .when(safe_col(silver_df, "status").isNull(), F.lit("missing_status"))
        .when(F.col("vehicle_movement_status") == "UNKNOWN", F.lit("unknown_movement"))
        .otherwise(F.lit("valid"))
    )
    .withColumn("silver_processed_at", F.current_timestamp())
)

display(silver_df.limit(20))

## 4. Write Silver Table

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_FQN)
)

print(f"Silver table created: {SILVER_FQN}")

## 5. Silver Validation

In [0]:
print("Movement status validation")
display(spark.sql(f"""
SELECT
  vehicle_movement_status,
  COUNT(*) AS total_rows
FROM {SILVER_FQN}
GROUP BY vehicle_movement_status
ORDER BY total_rows DESC
"""))

print("Data quality validation")
display(spark.sql(f"""
SELECT
  data_quality_status,
  COUNT(*) AS total_rows
FROM {SILVER_FQN}
GROUP BY data_quality_status
ORDER BY total_rows DESC
"""))

print("Daily row count validation")
display(spark.sql(f"""
SELECT
  transaction_date,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT assignment_id_value) AS total_transaction,
  COUNT(DISTINCT truck_id_value) AS total_vehicle,
  SUM(tonnage_value) AS total_tonnage
FROM {SILVER_FQN}
GROUP BY transaction_date
ORDER BY transaction_date DESC
LIMIT 30
"""))

## 6. Build Gold - Daily Summary

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_DAILY_FQN} AS
SELECT
  transaction_date,
  queue_name,
  port,
  truck_type,

  COUNT(DISTINCT assignment_id_value) AS total_transaction,
  COUNT(DISTINCT truck_id_value) AS total_vehicle,

  SUM(CASE WHEN vehicle_in_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS total_vehicle_in,
  SUM(CASE WHEN vehicle_out_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS total_vehicle_out,

  SUM(CASE WHEN status = 'COMPLETED' THEN 1 ELSE 0 END) AS completed_transaction,

  SUM(tonnage_value) AS total_tonnage,
  AVG(tonnage_value) AS avg_tonnage_per_vehicle,

  AVG(queue_duration_minutes) AS avg_queue_duration_minutes,
  AVG(cp_area_duration_minutes) AS avg_cp_area_duration_minutes,
  AVG(cp_process_duration_minutes) AS avg_cp_process_duration_minutes,

  SUM(CASE WHEN is_anomaly_bool = true THEN 1 ELSE 0 END) AS anomaly_count,
  SUM(CASE WHEN is_anomaly_queue_lane_bool = true THEN 1 ELSE 0 END) AS anomaly_queue_lane_count,
  SUM(CASE WHEN data_quality_status != 'valid' THEN 1 ELSE 0 END) AS data_quality_issue_count,

  CURRENT_TIMESTAMP() AS gold_processed_at

FROM {SILVER_FQN}
GROUP BY
  transaction_date,
  queue_name,
  port,
  truck_type
""")

print(f"Gold daily table created: {GOLD_DAILY_FQN}")

## 7. Build Gold - Hourly Summary

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_HOURLY_FQN} AS
SELECT
  transaction_date,
  transaction_hour,
  queue_name,
  port,

  COUNT(DISTINCT assignment_id_value) AS total_transaction,
  COUNT(DISTINCT truck_id_value) AS total_vehicle,

  SUM(CASE WHEN vehicle_in_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS vehicle_in,
  SUM(CASE WHEN vehicle_out_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS vehicle_out,

  SUM(tonnage_value) AS total_tonnage,
  AVG(tonnage_value) AS avg_tonnage_per_vehicle,

  AVG(queue_duration_minutes) AS avg_queue_duration_minutes,
  AVG(cp_area_duration_minutes) AS avg_cp_area_duration_minutes,
  AVG(cp_process_duration_minutes) AS avg_cp_process_duration_minutes,

  CURRENT_TIMESTAMP() AS gold_processed_at

FROM {SILVER_FQN}
GROUP BY
  transaction_date,
  transaction_hour,
  queue_name,
  port
""")

print(f"Gold hourly table created: {GOLD_HOURLY_FQN}")

## 8. Build Gold - Lane Summary

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_LANE_FQN} AS
SELECT
  transaction_date,
  queue_name,
  lane_id_value,
  lane_sb,

  COUNT(DISTINCT assignment_id_value) AS total_transaction,
  COUNT(DISTINCT truck_id_value) AS total_vehicle,

  SUM(CASE WHEN vehicle_in_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS vehicle_in,
  SUM(CASE WHEN vehicle_out_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS vehicle_out,

  SUM(tonnage_value) AS total_tonnage,
  AVG(tonnage_value) AS avg_tonnage_per_vehicle,

  AVG(cp_process_duration_minutes) AS avg_cp_process_duration_minutes,
  AVG(cp_area_duration_minutes) AS avg_cp_area_duration_minutes,

  SUM(CASE WHEN is_anomaly_bool = true THEN 1 ELSE 0 END) AS anomaly_count,
  SUM(CASE WHEN data_quality_status != 'valid' THEN 1 ELSE 0 END) AS data_quality_issue_count,

  CURRENT_TIMESTAMP() AS gold_processed_at

FROM {SILVER_FQN}
GROUP BY
  transaction_date,
  queue_name,
  lane_id_value,
  lane_sb
""")

print(f"Gold lane table created: {GOLD_LANE_FQN}")

## 9. Build Gold - Exception Monitoring

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_EXCEPTION_FQN} AS
SELECT
  transaction_date,
  transaction_ts,
  created_at_ts,
  queue_name,
  port,
  lane_id_value,
  lane_sb,
  truck_id_value,
  nomor_lambung,
  truck_type,
  status,
  completed_by,
  assigned_by,
  vehicle_movement_status,
  tonnage_value,
  sk_number,
  data_quality_status,
  is_anomaly_bool,
  is_anomaly_queue_lane_bool,
  tonnage_recovery_status,
  queue_duration_minutes,
  cp_area_duration_minutes,
  cp_process_duration_minutes,
  audit_latency_minutes,
  CURRENT_TIMESTAMP() AS gold_processed_at

FROM {SILVER_FQN}
WHERE
  data_quality_status != 'valid'
  OR vehicle_movement_status != 'IN_OUT_COMPLETED'
  OR is_anomaly_bool = true
  OR is_anomaly_queue_lane_bool = true
""")

print(f"Gold exception table created: {GOLD_EXCEPTION_FQN}")

## 10. Gold Validation

In [0]:
gold_tables = [
    GOLD_DAILY_FQN,
    GOLD_HOURLY_FQN,
    GOLD_LANE_FQN,
    GOLD_EXCEPTION_FQN,
]

for table_name in gold_tables:
    count_value = spark.table(table_name).count()
    print(f"{table_name}: {count_value:,} rows")

## 11. Dashboard Query Samples

Query di bawah bisa dipakai untuk membuat Databricks Dashboard.

In [0]:
# KPI Summary
kpi_query = f"""
SELECT
  COUNT(DISTINCT assignment_id_value) AS total_transaction,
  COUNT(DISTINCT truck_id_value) AS total_vehicle,

  SUM(CASE WHEN vehicle_in_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS total_vehicle_in,
  SUM(CASE WHEN vehicle_out_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS total_vehicle_out,

  SUM(CASE WHEN status = 'COMPLETED' THEN 1 ELSE 0 END) AS completed_transaction,

  SUM(tonnage_value) AS total_tonnage,
  AVG(tonnage_value) AS avg_tonnage_per_vehicle,

  AVG(queue_duration_minutes) AS avg_queue_duration_minutes,
  AVG(cp_area_duration_minutes) AS avg_cp_area_duration_minutes,
  AVG(cp_process_duration_minutes) AS avg_cp_process_duration_minutes,

  SUM(CASE WHEN is_anomaly_bool = true THEN 1 ELSE 0 END) AS anomaly_count,
  SUM(CASE WHEN data_quality_status != 'valid' THEN 1 ELSE 0 END) AS data_quality_issue_count

FROM {SILVER_FQN}
"""

display(spark.sql(kpi_query))

In [0]:
# Trend Kendaraan Masuk-Keluar per Jam
trend_query = f"""
SELECT
  transaction_date,
  transaction_hour,
  SUM(CASE WHEN vehicle_in_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS vehicle_in,
  SUM(CASE WHEN vehicle_out_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS vehicle_out,
  SUM(tonnage_value) AS total_tonnage
FROM {SILVER_FQN}
GROUP BY
  transaction_date,
  transaction_hour
ORDER BY
  transaction_date,
  transaction_hour
"""

display(spark.sql(trend_query))

In [0]:
# Output per CP
output_per_cp_query = f"""
SELECT
  queue_name,
  COUNT(DISTINCT assignment_id_value) AS total_transaction,
  COUNT(DISTINCT truck_id_value) AS total_vehicle,
  SUM(CASE WHEN vehicle_in_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS vehicle_in,
  SUM(CASE WHEN vehicle_out_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS vehicle_out,
  SUM(tonnage_value) AS total_tonnage,
  AVG(cp_area_duration_minutes) AS avg_cp_area_duration_minutes
FROM {SILVER_FQN}
GROUP BY queue_name
ORDER BY total_vehicle DESC
"""

display(spark.sql(output_per_cp_query))

In [0]:
# Lane Performance
lane_performance_query = f"""
SELECT
  queue_name,
  lane_id_value,
  lane_sb,
  COUNT(DISTINCT assignment_id_value) AS total_transaction,
  COUNT(DISTINCT truck_id_value) AS total_vehicle,
  SUM(CASE WHEN vehicle_in_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS vehicle_in,
  SUM(CASE WHEN vehicle_out_time_ts IS NOT NULL THEN 1 ELSE 0 END) AS vehicle_out,
  SUM(tonnage_value) AS total_tonnage,
  AVG(cp_process_duration_minutes) AS avg_cp_process_duration_minutes
FROM {SILVER_FQN}
GROUP BY
  queue_name,
  lane_id_value,
  lane_sb
ORDER BY
  queue_name,
  lane_id_value
"""

display(spark.sql(lane_performance_query))

## 12. Optional Optimization

Bagian ini opsional. Jalankan jika tabel sudah besar dan ingin mempercepat query dashboard.

In [0]:
for table_name in [SILVER_FQN, GOLD_DAILY_FQN, GOLD_HOURLY_FQN, GOLD_LANE_FQN, GOLD_EXCEPTION_FQN]:
    try:
        spark.sql(f"OPTIMIZE {table_name}")
        print(f"Optimized: {table_name}")
    except Exception as exc:
        print(f"Skip optimize for {table_name}: {exc}")

## 13. Final Output Tables

Setelah notebook berhasil dijalankan, tabel yang tersedia adalah:

```sql
`databricks-phase-a`.`default`.`silver_uc05_cp_vehicle_movement`
`databricks-phase-a`.`default`.`gold_uc05_cp_vehicle_daily_summary`
`databricks-phase-a`.`default`.`gold_uc05_cp_vehicle_hourly_summary`
`databricks-phase-a`.`default`.`gold_uc05_cp_vehicle_lane_summary`
`databricks-phase-a`.`default`.`gold_uc05_cp_vehicle_exception_monitoring`
```